In [ ]:
import re
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Specify the path to your text file in Google Drive
# Change this path to match your exact folder structure and filename
file_path = '/content/drive/MyDrive/physics_212_syllabus(example).txt'

try:
    with open(file_path, 'r', encoding='utf-8') as file:
        syllabus_text = file.read()
    print(f"Successfully loaded file from: {file_path}")
except FileNotFoundError:
    print(f"Error: Could not find the file at '{file_path}'. Please verify your file path.")
    syllabus_text = ""

# 3. Split into clean paragraph list
syllabus_paragraphs = [p.strip() for p in syllabus_text.strip().split('\n\n') if p.strip()]
print(f"Dataset successfully parsed: {len(syllabus_paragraphs)} paragraphs found.\n")

# ==========================================
# STRATEGY 1: Fixed Length Chunking (Sentence-Boundary Aware)
# ==========================================
def fixed_length_chunking(paragraphs, max_chars_per_chunk=400):
    chunks = []
    current_chunk = ""

    def split_sentences(text):
        return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

    for para in paragraphs:
        sentences = split_sentences(para)
        for sentence in sentences:
            if len(current_chunk) + len(sentence) + 1 > max_chars_per_chunk and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = sentence
            else:
                current_chunk += (" " + sentence) if current_chunk else sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

fixed_chunks = fixed_length_chunking(syllabus_paragraphs, max_chars_per_chunk=450)
print(f"--- Strategy 1: Fixed Length Chunking ({len(fixed_chunks)} chunks created) ---")
if fixed_chunks:
    print(f"Sample Chunk 1:\n{fixed_chunks[0]}\n")

# ==========================================
# STRATEGY 2: Overlapping Paragraph Chunking
# ==========================================
def overlapping_paragraph_chunking(paragraphs):
    chunks = []
    for i in range(len(paragraphs) - 1):
        combined = f"{paragraphs[i]}\n\n{paragraphs[i+1]}"
        chunks.append({
            "chunk_id": i + 1,
            "source_paragraphs": f"Paragraph {i+1} & {i+2}",
            "text": combined
        })
    return chunks

overlapping_chunks = overlapping_paragraph_chunking(syllabus_paragraphs)
print(f"--- Strategy 2: Overlapping Paragraph Chunking ({len(overlapping_chunks)} chunks created) ---")
if overlapping_chunks:
    print(f"Sample Chunk 1 ({overlapping_chunks[0]['source_paragraphs']}):\n{overlapping_chunks[0]['text'][:300]}...\n")

# ==========================================
# STRATEGY 3: Hybrid/Specialized Chunking (Currently working on it)
# ==========================================

Mounted at /content/drive
Successfully loaded file from: /content/drive/MyDrive/physics_212_syllabus.txt
Dataset successfully parsed: 30 paragraphs found.

--- Strategy 1: Fixed Length Chunking (52 chunks created) ---
Sample Chunk 1:
Welcome to Physics 212, an intensive and comprehensive exploration into the fundamental principles of electricity and magnetism. Throughout this course, students will investigate the physical laws governing electric charges, electric fields, magnetic fields, electromagnetic induction, and Maxwell's equations.

--- Strategy 2: Overlapping Paragraph Chunking (29 chunks created) ---
Sample Chunk 1 (Paragraph 1 & 2):
Welcome to Physics 212, an intensive and comprehensive exploration into the fundamental principles of electricity and magnetism. Throughout this course, students will investigate the physical laws governing electric charges, electric fields, magnetic fields, electromagnetic induction, and Maxwell's ...



#### 1. How Paragraphs Are Defined
We decided to define a paragraph to be something separated by a blank line. We also added code to make sure we do not count multiple empty lines as a paragraph by filtering out such occasions.

#### 2. How Overlap is Implemented

Adjacent Pairing: The for loop iterates through the list of paragraphs stopping one index short (len(paragraphs) - 1) so it can always look ahead to the next paragraph.

Mechanism:

Chunk 1 combines Paragraph 1 and Paragraph 2 (P1 + P2)

Chunk 2 combines Paragraph 2 and Paragraph 3 (P2 + P3)

Chunk 3 combines Paragraph 3 and Paragraph 4 (P3 + P4)

Etc etc


Ensures that if a search query or concept spans across a paragraph break, the semantic context isn't lost or cut off mid-thought.

In [ ]:
# Install required libraries if not already present in your Colab environment
# !pip install sentence-transformers faiss-cpu
!pip install faiss-cpu sentence-transformers transformers accelerate
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Assuming 'fixed_chunks' or 'hybrid_chunks' are already defined in your notebook context
# Fallback sample check if running standalone:
if 'fixed_chunks' not in locals():
    fixed_chunks = [
        "Welcome to Physics 212, focusing on electricity, magnetism, Maxwell's equations, and calculus-based physics.",
        "Weekly homework assignments are due every Thursday right before class at 12:00 PM, excluding weeks 1 and 16.",
        "The mid-term examination takes place during Week 8 in Lecture Hall 210, covering electrostatics and DC circuits."
    ]

print(f"Loaded {len(fixed_chunks)} text chunks for embedding conversion.\n")

# ==========================================
# 1. Define the 3 Hugging Face Models
# ==========================================
model_configs = {
    "Small": "all-MiniLM-L6-v2",         # ~384 dimensions
    "Medium": "all-mpnet-base-v2",       # ~768 dimensions
    "Large": "BAAI/bge-large-en-v1.5"    # ~1024-1536 dimensions (1024 dim)
}

vector_databases = {}

# ==========================================
# 2. Convert Chunks to Embeddings & Index in FAISS
# ==========================================
for tier, model_name in model_configs.items():
    print(f"Loading [{tier}] model -> {model_name}...")
    model = SentenceTransformer(model_name)

    print(f"Generating vector embeddings for {len(fixed_chunks)} chunks...")
    # Generate normalized embeddings for cosine similarity search
    embeddings = model.encode(fixed_chunks, convert_to_numpy=True, normalize_embeddings=True)

    dimension = embeddings.shape[1]

    # Create a searchable FAISS structure (IndexFlatIP uses Inner Product,
    # which equals Cosine Similarity since our vectors are normalized)
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    # Save references for searching
    vector_databases[tier] = {
        "model": model,
        "index": index,
        "embeddings": embeddings
    }
    print(f"[{tier}] Index built successfully! Vector Dimension: {dimension}\n")

# ==========================================
# 3. Test the Searchable Structures
# ==========================================
query = "When is the weekly homework due?"
print(f"==================================================")
print(f"TEST QUERY: '{query}'")
print(f"==================================================\n")

for tier, db in vector_databases.items():
    model = db["model"]
    index = db["index"]

    # Encode the search query with the same model
    query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)

    # Search for the top 1 most relevant chunk
    scores, indices = index.search(query_vector, k=1)

    best_idx = indices[0][0]
    best_score = scores[0][0]
    matched_text = fixed_chunks[best_idx]

    print(f"[{tier} Model] (Similarity Score: {best_score:.4f}):")
    print(f"   -> Result: \"{matched_text}\"\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 100.0 MB/s  0:00:00
Loaded 52 text chunks for embedding conversion.

Loading [Small] model -> all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Generating vector embeddings for 52 chunks...
[Large] Index built successfully! Vector Dimension: 1024

TEST QUERY: 'When is the weekly homework due?'

[Small Model] (Similarity Score: 0.7788):
   -> Result: "To reinforce classroom learning, there will be exactly one major homework assignment due every week. In strict accordance with course guidelines, these assignments are always due on Thursday right before class begins, specifically at 12:00 PM. Please note that weekly assignments are explicitly excluded during the first week of the semester and the final week of the semester."

[Medium Model] (Similarity Score: 0.7847):
   -> Result: "To reinforce classroom learning, there will be exactly one major homework assignment due every week. In strict accordance with course guidelines, these assignments are always due on Thursday right before class begins, specifically at 12:00 PM. Please note that weekly assignments are explicitly excluded during the first week of the semester and the fin

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# 1. Ensure we use fixed_chunks from Fixed Length Chunking
# ==========================================
# Double-check that fixed_chunks exists from your earlier cell
if 'fixed_chunks' not in locals():
    fixed_chunks = fixed_length_chunking(syllabus_paragraphs, max_chars_per_chunk=450)

print(f"Using {len(fixed_chunks)} chunks generated from fixed_length_chunking for embedding.\n")

# ==========================================
# 2. Embed fixed_chunks across Small, Medium, Large models
# ==========================================
vector_databases = {}

for tier, model_name in model_configs.items():
    print(f"Loading [{tier}] model -> {model_name}...")
    model = SentenceTransformer(model_name)

    # Encode fixed_chunks into normalized vector embeddings
    embeddings = model.encode(fixed_chunks, convert_to_numpy=True, normalize_embeddings=True)
    dimension = embeddings.shape[1]

    # Build FAISS Index (IndexFlatIP computes Cosine Similarity for normalized vectors)
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    vector_databases[tier] = {
        "model": model,
        "index": index,
        "embeddings": embeddings
    }
    print(f"[{tier}] Index built successfully with {len(fixed_chunks)} fixed-length chunks!\n")

# ==========================================
# 3. Define Top-k Retrieval using fixed_chunks
# ==========================================
def retrieve_top_k(query, model, faiss_index, chunks, k=3):
    query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = faiss_index.search(query_vector, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append({
                "score": float(score),
                "chunk_index": int(idx),
                "text": chunks[idx]  # Pulls directly from fixed_chunks
            })
    return results

# ==========================================
# 4. Load Qwen LLM Generation Model
# ==========================================
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading LLM [{MODEL_ID}]...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("LLM loaded successfully!\n")

# ==========================================
# 5. Prompt Construction & Generation Function
# ==========================================
def generate_rag_response(system_instruction, retrieved_results, user_query):
    context_blocks = []
    for res in retrieved_results:
        block = f"[Fixed-Length Chunk {res['chunk_index']} (Score: {res['score']:.4f})]:\n{res['text']}"
        context_blocks.append(block)

    retrieved_context_text = "\n\n".join(context_blocks)

    user_content = f"""Here is the retrieved context from the course syllabus (fixed-length chunks):
{retrieved_context_text}

Based strictly on the provided context above, please answer the following question:
{user_query}"""

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content}
    ]

    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([formatted_text], return_tensors="pt").to(llm_model.device)

    print("Generating response from Qwen...")
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True
        )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# ==========================================
# 6. Execute RAG Pipeline (Select Tier & Query)
# ==========================================
system_instruction = (
    "You are an expert academic assistant for Physics 212: Electricity and Magnetism. "
    "Answer the user's question accurately and concisely using ONLY the provided "
    "retrieved context from the fixed-length syllabus chunks. Do not assume or extrapolate facts."
)

user_query = "When are the weekly homework assignments due, and when does the midterm exam take place?"

# Select your desired embedding model tier here: "Small", "Medium", or "Large"
selected_tier = "Medium"
print(f"Running retrieval using [{selected_tier}] tier model over fixed-length chunks...")

retriever_model = vector_databases[selected_tier]["model"]
faiss_index = vector_databases[selected_tier]["index"]

# Retrieve top-3 chunks from fixed_chunks
top_retrieved_chunks = retrieve_top_k(user_query, retriever_model, faiss_index, fixed_chunks, k=3)

# Generate final answer
final_answer = generate_rag_response(system_instruction, top_retrieved_chunks, user_query)

print("\n==================================================")
print(f"USER QUERY: {user_query}")
print("==================================================")
print(final_answer)


Using 52 chunks generated from fixed_length_chunking for embedding.

Loading [Small] model -> all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Small] Index built successfully with 52 fixed-length chunks!

Loading [Medium] model -> all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[Medium] Index built successfully with 52 fixed-length chunks!

Loading [Large] model -> BAAI/bge-large-en-v1.5...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[Large] Index built successfully with 52 fixed-length chunks!

Loading LLM [Qwen/Qwen3-4B-Instruct-2507]...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LLM loaded successfully!

Running retrieval using [Medium] tier model over fixed-length chunks...
Generating response from Qwen...

USER QUERY: When are the weekly homework assignments due, and when does the midterm exam take place?
The weekly homework assignments are due on Thursday at 12:00 PM, right before class begins. The midterm exam takes place during week eight.
